# DSA_01 — Available Origins & Application Data Contract

**Purpose.** Discover forecast origins from the project's actual operational
files. No date is typed or invented manually.

In [1]:
# Import libraries
from pathlib import Path
import sys, yaml, pandas as pd

In [3]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG = PROJECT_ROOT / "configs" / "decision_support_app.yaml"
CONFIG


WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/decision_support_app.yaml')

In [4]:
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))

In [5]:
# Import modules for decision_support app
from src.ontario_peak_risk.decision_support.common import resolve
from src.ontario_peak_risk.decision_support.availability import (
    discover_available_origins,
    choose_demo_origins,
)

In [ ]:
available = discover_available_origins(cfg, PROJECT_ROOT)
if available.empty:
    raise ValueError(
        "No forecast origin currently satisfies both the 168-hour demand "
        "history requirement and the h0...h24 weather-grid requirement."
    )

available_path = resolve(PROJECT_ROOT, cfg["paths"]["available_origins"])
available_path.parent.mkdir(parents=True, exist_ok=True)
available.to_csv(available_path, index=False)

print("Valid origins:", len(available))
print("From:", available["forecast_origin"].min())
print("To:", available["forecast_origin"].max())
display(available.head())

Valid origins: 2712
From: 2026-01-08 00:00:00
To: 2026-04-30 23:00:00


,forecast_origin,demand_history_status,weather_grid_status,fsa_count
0,2026-01-08 00:00:00,PASS,PASS,6
1,2026-01-08 01:00:00,PASS,PASS,6
2,2026-01-08 02:00:00,PASS,PASS,6
3,2026-01-08 03:00:00,PASS,PASS,6
4,2026-01-08 04:00:00,PASS,PASS,6


In [8]:
demo_origins = choose_demo_origins(
    available,
    max_origins=cfg["demo"]["max_origins"],
    strategy=cfg["demo"]["selection_strategy"],
)

demo_path = resolve(PROJECT_ROOT, cfg["paths"]["demo_origins"])
demo_origins.to_csv(demo_path, index=False)

In [9]:
print("Selected demo origins:")
display(demo_origins)

Selected demo origins:


,forecast_origin,demand_history_status,weather_grid_status,fsa_count
0,2026-01-08 00:00:00,PASS,PASS,6
1,2026-03-05 12:00:00,PASS,PASS,6
2,2026-04-30 23:00:00,PASS,PASS,6


- The generated `demo_origins.csv` is the auditable source for the dates shown in the public application. 
- You may replace those rows with other origins from `available_origins.csv`, but only validated origins should be used.